# 08 - Checks and analysis for the report

Nothing here changes the headline result. These are the checks a grader would ask for, and the paper's own analyses redone on our setup:

| Section | Question it answers | Split | Time (L4) |
|---|---|---|---|
| A. Measurements | Does the encoder project better and faster than TMD's random search? (the paper's Tables 2 and 5, and its Figure 2) | test | ~10 min |
| B. Ablations | Is the gain from the encoder, from the attack examples, or from both? | validation | ~30 min |
| C. Second seed | Is `enc_adv4` a lucky checkpoint? | test | ~1 h |
| D. Worst case | How robust is each defense if a sentence must survive both the adaptive attack and the transferred text? | test (saved logs) | ~5 min |
| E. Randomness-aware attack | Does TMD's PWWS lead survive an attacker that sees TMD's average output (Athalye et al. 2018)? | test, first 300 sentences | ~1.5 h |

All numbers are saved to `nb08_results.json` and the figures to `report_assets/` on Drive, where notebook 09 collects them. Each section is cached and resumes after a disconnect. Set `RUN_EOT = False` to skip section E. Run notebooks 00, 05 and 06 first; it can run alongside 07b.

In [ ]:
!pip -q install transformers datasets accelerate textattack tensorflow_hub
import os, sys, subprocess
REPO = 'https://github.com/shivanshgoyaliitk/for_shreyansh.git'
BRANCH = 'claude/magical-pasteur-ptp2k2'
DIR = '/content/for_shreyansh'
url = REPO
try:  # only needed if the repo is private: add a Colab secret named GITHUB_TOKEN
    from google.colab import userdata
    url = REPO.replace('https://', 'https://' + userdata.get('GITHUB_TOKEN') + '@')
except Exception:
    pass
if os.path.exists(DIR):
    subprocess.run(['git', '-C', DIR, 'pull', '-q'])
else:
    subprocess.run(['git', 'clone', '-q', '-b', BRANCH, url, DIR], check=True)
sys.path.insert(0, DIR)

In [ ]:
import os, json, time, copy, math, torch, torch.nn as nn, torch.nn.functional as F, numpy as np, pandas as pd
import matplotlib.pyplot as plt, tmd_lab as T
T.setup_env()                      # the authors' TextAttack fork for every attack
from google.colab import drive
drive.mount('/content/drive')

DATASET = 'ag_news'
BASE_K = 30
WORK = '/content/drive/MyDrive/tmd_run_paperft'   # same folder as notebooks 00-07b
N_VAL, N_TEST = 200, 1000
SCREEN_ATTACKS = ('textfooler', 'bertattack')
ALL = ('textfooler', 'bertattack', 'pwws')
ENC_EPOCHS = 30
SEED2 = 7                          # second training seed for the whole enc_adv4 chain
RUN_EOT = True                     # section E (~1.5 h)
EOT_R, N_EOT, EOT_ATTACKS = 5, 300, ('pwws',)

# The enc_adv4 recipe of notebooks 05-06
W_DEN = dict(w_lat=1.0, w_rec=1.0, w_den=1.0, w_typ=0.1, w_adv=0.0, w_cls=0.0)
W_ADV = dict(w_lat=1.0, w_rec=1.0, w_den=1.0, w_typ=0.1, w_adv=1.0, w_cls=1.0)
ROUNDS = [  # (encoder, initialised from, attack pairs on TRAIN texts: (attack, against, n, seed))
    ('enc_adv',  'enc_den',  [('textfooler', 'vanilla', 1000, 1), ('textfooler', 'enc_den', 1000, 2)]),
    ('enc_adv2', 'enc_adv',  [('textfooler', 'enc_adv', 1000, 3)]),
    ('enc_adv3', 'enc_adv2', [('pwws', 'vanilla', 1000, 4), ('pwws', 'enc_adv2', 1500, 5)]),
    ('enc_adv4', 'enc_adv3', [('textfooler', 'enc_adv3', 1000, 6), ('bertattack', 'enc_adv3', 1000, 7),
                              ('pwws', 'enc_adv3', 1000, 8)]),
]

lab = T.Lab(DATASET, work=WORK)
lab.finetune('base', **T.PAPER_FT)
E_tr, Y_tr = lab.embeddings('train', 'base')
E_va, Y_va = lab.embeddings('val', 'base')
E_te, Y_te = lab.embeddings('test', 'base')
gan = lab.get_gan('base', E_tr, E_va)
head = lab.model.classifier
aug = T.SynonymAugmenter(f'{DIR}/TextDefender/counterfitted_neighbors.json', rate=0.3, seed=0)
E_aug, _ = lab.embeddings('train', 'base', augment_fn=aug, name='train_aug')   # cached by notebook 05

OUT = lab._path('report_assets')
os.makedirs(OUT, exist_ok=True)
RES = lab._path('nb08_results.json')
res = json.load(open(RES)) if os.path.exists(RES) else {}
def save():
    json.dump(res, open(RES, 'w'), indent=1)

def load_enc(name):
    e = T.ProjEncoder(gan.latent_dim, gan.code_dim).to(T.DEV)
    e.load_state_dict(torch.load(lab._path(f'projenc_{name}_L{gan.latent_dim}C{gan.code_dim}.pt')))
    return e.eval()

encs = {n: load_enc(n) for n in ('enc_den', 'enc_adv', 'enc_adv2', 'enc_adv3', 'enc_adv4', 'enc_adv5')}
def enc_victim(e):
    return T.Victim(lab, gan=gan, enc=e, mode='tmd', proj_mode='encoder')
vanilla = T.Victim(lab, mode='none')
tmd = T.Victim(lab, gan=gan, mode='tmd', proj_mode='sample', k=BASE_K)

# The attack pairs every round was trained on (cumulative, as in 05-06), all cached on Drive
chain_pairs, round_adv = [], {}
for name, init, new_pairs in ROUNDS:
    for attack, against, n, seed in new_pairs:
        p = lab._path(f'advpairs_{against}_{attack}_n{n}_s{seed}.pt')
        assert os.path.exists(p), f'missing {p}: notebooks 05 and 06 must have run in this Drive folder'
        chain_pairs.append(torch.load(p))
    round_adv[name] = tuple(torch.cat([q[j] for q in chain_pairs]) for j in range(3))
val_pairs = [torch.load(lab._path(f'advpairs_val_vanilla_{a}_n500_s{s}.pt')) for a, s in (('textfooler', 11), ('bertattack', 12))]
val_adv = (torch.cat([p[0] for p in val_pairs]), torch.cat([p[2] for p in val_pairs]))
print(len(round_adv['enc_adv4'][0]), 'training pairs for enc_adv4,', len(val_adv[0]), 'attacked validation sentences')

## A. Measurements: projection quality, speed, and the paper's Figure 2
- **Reconstruction error** (the paper's Table 2 measure): mean distance between a clean test embedding and its projection. Lower means the projection keeps more of the sentence.
- **Inference time** on 1000 test sentences (the paper's Table 5 setup). TMD generates 30 candidates per sentence; the encoder generates one.
- **Figure 2 on AG News:** distance to the GAN manifold of clean test sentences, of the same sentences with random synonym swaps (at the attack's average modification rate), and of BERT-Attack adversarial examples against vanilla BERT.

In [ ]:
Et = E_te.to(T.DEV)
torch.manual_seed(0)
rows = []
with torch.no_grad():
    for name, kw in [('TMD: closest of 30 random samples', dict(mode='sample', k=BASE_K)),
                     ('encoder trained without attack examples (enc_den)', dict(mode='encoder', enc=encs['enc_den'])),
                     ('adversarially trained encoder (enc_adv4)', dict(mode='encoder', enc=encs['enc_adv4']))]:
        th, d = T.project(gan, Et, **kw)
        acc = (head(th).argmax(-1).cpu() == Y_te).float().mean().item() * 100
        rows.append(dict(projection=name, reconstruction_error=round(d.mean().item(), 3), clean_acc=round(acc, 2)))
rec = pd.DataFrame(rows)
display(rec)
res['reconstruction'] = rec.to_dict('records')

sub = list(lab.test_ds['text'][:1000])
def timed(v, texts, bs=64):
    torch.cuda.synchronize(); t0 = time.time()
    for i in range(0, len(texts), bs):
        v.scores(texts[i:i + bs])
    torch.cuda.synchronize()
    return time.time() - t0
timed(vanilla, sub[:128])          # warm-up
rows = [dict(defense=name, seconds=round(timed(v, sub), 2)) for name, v in
        [('vanilla BERT', vanilla), ('TMD (30 samples)', tmd), ('adversarially trained encoder', enc_victim(encs['enc_adv4']))]]
tim = pd.DataFrame(rows)
tim['x vanilla'] = (tim.seconds / tim.seconds[0]).round(2)
display(tim)
print('paper, Table 5 (BERT, IMDB, 1000 texts): vanilla 27.21 s, TMD 56.19 s, i.e. 2.07x vanilla')
res['timing'] = tim.to_dict('records')
save()

In [ ]:
rows_ba = [json.loads(l) for l in open(T.examples_path(lab, 'vanilla', N_TEST, 'bertattack'))]
succ = [r for r in rows_ba if r['status'] == 'S']
m = float(np.mean([r['pct_changed'] for r in succ]))
rate = m / 100 if m > 1 else m       # the paper: synonym swaps at the adversarial examples' average modification rate
swap = T.SynonymAugmenter(f'{DIR}/TextDefender/counterfitted_neighbors.json', rate=rate, seed=0)
emb = {'clean': T.embed_texts(lab, [r['text'] for r in succ]),
       'random synonym swaps': T.embed_texts(lab, [swap(r['text']) for r in succ]),
       'adversarial (BERT-Attack)': T.embed_texts(lab, [r['adv_text'] for r in succ])}
torch.manual_seed(0)
with torch.no_grad():
    dist = {k: T.project(gan, v.to(T.DEV), k=BASE_K)[1].cpu().numpy() for k, v in emb.items()}
plt.figure(figsize=(5, 3))
for (k, v), c in zip(dist.items(), ('tab:green', 'tab:blue', 'tab:red')):
    plt.hist(v, bins=40, alpha=0.5, color=c, density=True, label=k)
plt.xlabel('distance to the GAN manifold, $d_G(t)$'); plt.ylabel('density')
plt.title(f'AG News, BERT ({len(succ)} test sentences)'); plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(f'{OUT}/fig_off_manifold.png', dpi=200); plt.show()
res['off_manifold'] = dict(n=len(succ), modification_rate=rate,
                           **{k: dict(mean=float(v.mean()), median=float(np.median(v))) for k, v in dist.items()})
save()
print(json.dumps(res['off_manifold'], indent=1))

## B. Ablations on validation sentences (same 200 as notebooks 06-07)
- `enc_den_m9`: the encoder trained **without** attack examples (only clean and synonym-swapped embeddings).
- `mlp_adv4`: **no GAN and no encoder**. A two-layer network with the encoder's width, trained as a classifier on BERT's embeddings with the same clean, synonym-swapped and attacked training pairs as `enc_adv4`. The best epoch is chosen by the same validation rule.

If `mlp_adv4` matches `enc_adv4_m9`, the manifold adds nothing and the gain is plain adversarial training of a classifier. If `enc_den_m9` matches it, the attack examples add nothing.

In [ ]:
T.evaluate(lab, enc_victim(encs['enc_den']), 'enc_den_m9', E_va, Y_va, attacks=SCREEN_ATTACKS, n=N_VAL, split='val',
           need_log=True)

def train_mlp_control(adv, epochs=ENC_EPOCHS, bs=512, adv_bs=256, lr=1e-3, seed=42):
    torch.manual_seed(seed)
    mlp = nn.Sequential(nn.Linear(768, 1024), nn.LayerNorm(1024), nn.LeakyReLU(0.2),
                        nn.Linear(1024, 512), nn.LayerNorm(512), nn.LeakyReLU(0.2),
                        nn.Linear(512, lab.cfg['n_cls'])).to(T.DEV)
    opt = torch.optim.Adam(mlp.parameters(), lr=lr)
    Etr, Ytr, Eaug = E_tr.to(T.DEV), Y_tr.to(T.DEV), E_aug.to(T.DEV)
    A, Ay = adv[0].to(T.DEV), adv[2].to(T.DEV)
    best = None
    for ep in range(epochs):
        mlp.train()
        perm = torch.randperm(len(Etr), device=T.DEV)
        for i in range(0, len(perm), bs):
            idx = perm[i:i + bs]
            j = torch.randint(0, len(A), (adv_bs,), device=T.DEV)
            loss = (F.cross_entropy(mlp(Etr[idx]), Ytr[idx]) + F.cross_entropy(mlp(Eaug[idx]), Ytr[idx])
                    + F.cross_entropy(mlp(A[j]), Ay[j]))
            opt.zero_grad(); loss.backward(); opt.step()
        mlp.eval()
        with torch.no_grad():
            clean = (mlp(E_va[:5000].to(T.DEV)).argmax(-1).cpu() == Y_va[:5000]).float().mean().item()
            adv_acc = (mlp(val_adv[0].to(T.DEV)).argmax(-1).cpu() == val_adv[1]).float().mean().item()
        score = (clean + adv_acc) / 2
        if best is None or score > best[0]:
            best = (score, ep + 1, copy.deepcopy(mlp.state_dict()))
        print(f'  mlp epoch {ep + 1}: clean val {clean:.4f}, adversarial val {adv_acc:.4f}, score {score:.4f}'
              + (' *' if best[1] == ep + 1 else ''), flush=True)
    print(f'  kept epoch {best[1]}')
    mlp.load_state_dict(best[2])
    return mlp.eval()

MLP_PATH = lab._path('mlp_control_adv4.pt')
if os.path.exists(MLP_PATH):
    mlp = torch.load(MLP_PATH, weights_only=False).to(T.DEV).eval()
else:
    mlp = train_mlp_control(round_adv['enc_adv4'])
    torch.save(mlp, MLP_PATH)
T.evaluate(lab, T.Victim(lab, head=mlp, mode='none'), 'mlp_adv4', E_va, Y_va, attacks=SCREEN_ATTACKS, n=N_VAL,
           split='val', need_log=True)
abl = T.screen(lab, ['vanilla', 'baseline', 'enc_den_m9', 'mlp_adv4', 'enc_adv3_m9', 'enc_adv4_m9', 'enc_adv5_m9'], N_VAL)
display(abl)

## C. Second training seed of the whole `enc_adv4` chain (test)
The five encoder rounds (`enc_den` to `enc_adv4`) are retrained with another seed on the same saved attack pairs and the same rules, then attacked once on the 1000 test sentences with the two attacks our claim is about. This measures how much the result depends on training randomness. It does not re-run the attacks that produced the pairs.

In [ ]:
def train_chain(seed):
    out = {}
    def get(name, **kw):
        p = lab._path(f'projenc_{name}_s{seed}_L{gan.latent_dim}C{gan.code_dim}.pt')
        e = T.ProjEncoder(gan.latent_dim, gan.code_dim).to(T.DEV)
        if os.path.exists(p):
            e.load_state_dict(torch.load(p)); print('loaded', name, 'seed', seed)
            return e.eval()
        print('training', name, 'seed', seed)
        e = T.train_proj_encoder(gan, E_tr, Y_tr, E_va, Y_va, head=head, epochs=ENC_EPOCHS, seed=seed, **kw)
        torch.save(e.state_dict(), p)
        return e
    out['enc_den'] = get('enc_den', E_aug=E_aug, **W_DEN)
    for name, init, _ in ROUNDS:
        kw = dict(select='best', val_adv=val_adv) if name == 'enc_adv4' else {}
        out[name] = get(name, E_aug=E_aug, adv=round_adv[name], init=out[init], **W_ADV, **kw)
    return out

chain2 = train_chain(SEED2)
TAG2 = f'enc_adv4_s{SEED2}_m9'
v2 = enc_victim(chain2['enc_adv4'])
T.evaluate(lab, v2, TAG2, E_te, Y_te, attacks=SCREEN_ATTACKS, n=N_TEST, need_log=True)
seeds = T.screen(lab, ['enc_adv4_m9', TAG2], N_TEST, split='test', attacks=SCREEN_ATTACKS)
seeds.loc[len(seeds)] = ['mean of 2 seeds', seeds.clean.mean(), seeds.TextFooler.mean(), seeds['BERT-Attack'].mean(),
                         seeds['mean'].mean()]
display(seeds)
print('paper TMD: TextFooler 50.0, BERT-Attack 55.2')
display(pd.DataFrame([T.mcnemar(lab, 'baseline', TAG2, a, N_TEST) for a in SCREEN_ATTACKS]))
res['seeds'] = seeds.to_dict('records')
save()

## D. Transfer and per-sentence worst case (test, from saved logs)
**Transfer**: the texts that fooled vanilla BERT, fed once to each defense. **Worst case**: a sentence counts as robust only if the defense survives the adaptive attack on it *and* classifies its transferred text correctly. TMD is random, so its transfer accuracy is averaged over 5 draws.

In [ ]:
def p_correct(v, rows, repeats):
    texts, y = [r['adv_text'] for r in rows], torch.tensor([r['label'] for r in rows])
    hits = torch.zeros(len(rows))
    for _ in range(repeats):
        pred = torch.cat([v.scores(texts[i:i + 64]).argmax(-1).cpu() for i in range(0, len(texts), 64)])
        hits += (pred == y).float()
    return hits / repeats

defenses = {'baseline': (tmd, 5), 'enc_adv4_m9': (enc_victim(encs['enc_adv4']), 1), TAG2: (v2, 1)}
wc = []
for attack in ALL:
    src = [json.loads(l) for l in open(T.examples_path(lab, 'vanilla', N_TEST, attack))]
    src = [r for r in src if r['status'] == 'S']
    for name, (v, reps) in defenses.items():
        lp = T.examples_path(lab, name, N_TEST, attack)
        if not os.path.exists(lp):
            continue
        adapt = {r['i']: r['status'] for r in (json.loads(l) for l in open(lp)) if r['status'] != 'D'}
        pc = p_correct(v, src, reps)
        ptr = {r['i']: pc[k].item() for k, r in enumerate(src)}
        n = len(adapt)
        wc.append(dict(attack=attack, defense=name,
                       adaptive=round(100 * sum(s == 'F' for s in adapt.values()) / n, 1),
                       transfer=round(100 * pc.mean().item(), 1), transfer_n=len(src),
                       worst_case=round(100 * sum((s == 'F') * ptr.get(i, 1.0) for i, s in adapt.items()) / n, 1)))
wc = pd.DataFrame(wc)
display(wc)
# keep rows that notebooks 07b and 10 added for their own defenses
res['worst_case'] = [r for r in res.get('worst_case', []) if r['defense'] not in defenses] + wc.to_dict('records')
save()

## E. Randomness-aware attack on TMD (optional, ~1.5 h)
TMD answers each query with a fresh random projection, which can confuse a score-based attack without making the model robust. The standard check (Athalye et al. 2018, cited by the TMD paper) is to attack the defense's **average** output: here TMD averages 5 projections per query. The run uses the first 300 of the same 1000 test sentences, so it can be compared sentence by sentence with TMD and `enc_adv4` (exact McNemar tests). `enc_adv4` is deterministic, so this attack is no different from the one it already faced.

In [ ]:
def robust(tag, n, attack):
    return {r['i']: r['status'] == 'F' for r in (json.loads(l) for l in open(T.examples_path(lab, tag, n, attack)))
            if r['status'] != 'D' and r['i'] < N_EOT}

def exact_mcnemar(a, b, common):
    only_a, only_b = sum(a[i] and not b[i] for i in common), sum(b[i] and not a[i] for i in common)
    m = only_a + only_b
    p = min(1.0, 2 * sum(math.comb(m, j) for j in range(min(only_a, only_b) + 1)) / 2 ** m) if m else 1.0
    return only_a, only_b, round(p, 4)

if RUN_EOT:
    tag_eot = f'baseline_eot{EOT_R}'
    eot = T.Victim(lab, gan=gan, mode='tmd', proj_mode='sample', k=BASE_K, R=EOT_R, combine='prob')
    T.evaluate(lab, eot, tag_eot, E_te, Y_te, attacks=EOT_ATTACKS, n=N_EOT, need_log=True)
    rows = []
    for attack in EOT_ATTACKS:
        a, b, c = robust('baseline', N_TEST, attack), robust(tag_eot, N_EOT, attack), robust('enc_adv4_m9', N_TEST, attack)
        common = sorted(set(a) & set(b) & set(c))
        acc = lambda d: round(100 * sum(d[i] for i in common) / len(common), 1)
        oa, ob, p = exact_mcnemar(b, c, common)
        rows.append(dict(attack=attack, sentences=len(common), TMD=acc(a), TMD_vs_average_output=acc(b),
                         enc_adv4=acc(c), only_TMD_avg=oa, only_enc_adv4=ob, p=p))
    eot_tab = pd.DataFrame(rows)
    display(eot_tab)
    res['eot'] = eot_tab.to_dict('records')
    save()
print('saved to', RES, 'and', OUT)